# Q2b — BERT fine-tuning baseline

**W&B run:** _filled in after first run_

Single tuned fine-tuning run on `bert-base-uncased` with the hyperparameters that the report
treats as the Q2 baseline. The saved metrics here feed the cross-experiment comparison in q2d.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
from pathlib import Path

import numpy as np
import torch
import wandb

from nlp_project import SEED, set_seed
from nlp_project.bert_data import build_splits
from nlp_project.bert_train import run_finetune
from nlp_project.eval import plot_confusion

set_seed()
FIG_DIR = Path('../figures'); FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path('../models'); MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path('../models') / 'q2_results'; RESULTS_DIR.mkdir(exist_ok=True)

os.environ.setdefault('WANDB_PROJECT', 'hslu-nalapro')
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {DEVICE}')


device: mps


In [2]:
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 256
splits = build_splits(tokenizer_name=MODEL_NAME, max_length=MAX_LENGTH, seed=SEED)
print(f"train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")


train=10182, val=1132, test=7532


In [3]:
RUN_NAME = 'q2b-bert-base-uncased-baseline'
config = {
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'lr': 2e-5,
    'batch_size': 16,
    'epochs': 3,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'freeze_encoder': False,
    'seed': SEED,
}
run = wandb.init(project='hslu-nalapro', name=RUN_NAME, group='q2', config=config)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: danwwaititu (danwwaititu-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
result = run_finetune(
    splits=splits,
    out_dir=MODEL_DIR / 'q2b',
    model_name=MODEL_NAME,
    lr=config['lr'],
    epochs=config['epochs'],
    batch_size=config['batch_size'],
    run_name=RUN_NAME,
    weight_decay=config['weight_decay'],
    warmup_ratio=config['warmup_ratio'],
    report_to=['wandb'],
)
metrics = result['test_metrics']
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test macro-F1: {metrics['macro_f1']:.4f}")
print(f"best val macro-F1: {result['best_eval_metric']:.4f}")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.074102,1.014878,0.696113,0.674454
2,0.774316,0.847147,0.752650,0.732609
3,0.592223,0.823264,0.758834,0.744466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

test accuracy: 0.7080
test macro-F1: 0.6888
best val macro-F1: 0.7445


In [5]:
plot_confusion(
    metrics['confusion_matrix'], splits['label_names'],
    save_path=FIG_DIR / 'q2b_confusion_matrix.png',
    title='Q2b — bert-base-uncased baseline',
)
run.log({
    'test_accuracy': metrics['accuracy'],
    'test_macro_f1': metrics['macro_f1'],
})
run.finish()


eval/accuracy,▁▇█
eval/loss,█▂▁
eval/macro_f1,▁▇█
eval/runtime,▆█▁
eval/samples_per_second,▃▁█
eval/steps_per_second,▃▁█
test/accuracy,▁
test/loss,▁
test/macro_f1,▁
test/runtime,▁
+9,...


In [6]:
# Persist the baseline metrics for q2d to pick up.
out = {
    'name': RUN_NAME,
    'config': config,
    'accuracy': float(metrics['accuracy']),
    'macro_f1': float(metrics['macro_f1']),
    'per_class_f1': metrics['per_class_f1'].tolist(),
    'confusion_matrix': metrics['confusion_matrix'].tolist(),
}
(RESULTS_DIR / 'q2b_baseline.json').write_text(json.dumps(out, indent=2))
print('saved', RESULTS_DIR / 'q2b_baseline.json')


saved ../models/q2_results/q2b_baseline.json


## Notes for the report

- This run is the Q2 baseline. Compare against Q1's TF-IDF and word2vec-mean numbers in q2d.
- Hyperparameters were not tuned at this stage; q2c sweeps them.
